# 模型类

到目前为止，我们已经构建了五个核心基础类：**张量**、**层**、**损失函数**、**优化器**、**数据集**，它们各自负责一件事。本章引入最后一个核心类：**模型**（Model），它把层、损失函数、优化器，以及完整的训练和测试流程全部封装在一起。

```{figure} images/model.png
:align: center
:width: 360px
**图例：框架分层结构**

* **顶层**：模型（流程控制）；
* **中层**：层、损失函数、优化器（逻辑执行）；
* **底层**：张量（自动微分）。

```

In [1]:
from abc import ABC, abstractmethod

import numpy as np

## 基础架构

### 张量

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for parent in self.parents:
            parent.backward()

    @property
    def shape(self):
        return self.data.shape

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础数据集

In [3]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size

        self.test_features = None
        self.test_labels = None
        self.train_features = None
        self.train_labels = None

        self.load()

        self.features = self.train_features
        self.labels = self.train_labels

    @abstractmethod
    def load(self):
        pass

    def train(self):
        self.features = self.train_features
        self.labels = self.train_labels

    def eval(self):
        self.features = self.test_features
        self.labels = self.test_labels

    def items(self):
        return Tensor(self.features), Tensor(self.labels)

    def __len__(self):
        return len(self.features) // self.batch_size

    def __getitem__(self, index):
        start = index * self.batch_size
        end = start + self.batch_size

        feature = Tensor(self.features[start: end])
        label = Tensor(self.labels[start: end])
        return feature, label

### 基础层

In [4]:
class Layer(ABC):

    def __call__(self, x: Tensor):
        return self.forward(x)

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    @property
    def parameters(self):
        return []

    def __repr__(self):
        return f"{type(self).__name__}[]"

### 基础损失函数

In [5]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor):
        pass

### 基础优化器

In [6]:
class Optimizer(ABC):

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def reset(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

    @abstractmethod
    def step(self):
        pass

### 基础模型

**基础模型**（Base Model）是一个抽象类，定义了模型的两个核心接口：

* **训练函数 `train()`**：训练模型；
* **测试函数 `test()`**：评估模型。

创建一个模型需要提供三样东西：**层**、**损失函数**、**优化器**。

In [7]:
class Model(ABC):

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    @abstractmethod
    def train(self, dataset, epochs):
        pass

    @abstractmethod
    def test(self, dataset):
        pass

## 数据

### 数据集（冰激凌销量）

In [8]:
class IcecreamDataset(Dataset):

    def load(self):
        self.train_features = [[22.5, 72.0],
                               [31.4, 45.0],
                               [19.8, 85.0],
                               [27.6, 63.0]]
        self.train_labels = [[95],
                             [210],
                             [70],
                             [155]]

        self.test_features = [[28.1, 58.0]]
        self.test_labels = [[165]]

## 模型

### 线性层

In [9]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        self.weight = Tensor(np.ones((out_size, in_size)) / in_size)
        self.bias = Tensor(np.zeros(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)

        p.gradient_fn = gradient_fn
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

    def __repr__(self):
        return f'Linear[weight{self.weight.shape}; bias{self.bias.shape}]'

### 均方误差损失函数

In [10]:
class MSELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            p.grad += -2 * (y.data - p.data) / len(y.data)

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

### 随机梯度下降优化器

In [11]:
class SGDOptimizer(Optimizer):

    def step(self):
        for param in self.parameters:
            param.data -= param.grad * self.lr

### 神经网络模型

`NNModel` 是第一个具体的模型类，把完整的训练和测试流程封装在内。

**训练函数 `train()`** 实现了标准的训练循环：外层按 `epochs` 轮次迭代，内层按批次遍历训练集。


**测试函数 `test()`** 切换到测试模式，用全部测试集做一次前向传播，返回预测值和测试损失。

In [12]:
class NNModel(Model):

    def train(self, dataset, epochs):
        # 切换到训练模式
        dataset.train()

        for epoch in range(epochs):
            epoch_loss = 0
            for i in range(len(dataset)):
                features, labels = dataset[i]

                # 1. 梯度清零
                self.optimizer.reset()
                # 2. 前向传播
                predictions = self.layer(features)
                # 3. 计算损失
                loss = self.loss_fn(predictions, labels)
                # 4. 梯度计算（自动微分）
                loss.backward()
                # 5. 更新参数
                self.optimizer.step()
                epoch_loss += float(loss.data)

    def test(self, dataset):
        # 切换到测试模式
        dataset.eval()

        features, labels = dataset.items()
        predictions = self.layer(features)
        loss = self.loss_fn(predictions, labels)
        return predictions, loss

## 训练

### 学习率

In [13]:
LEARNING_RATE = 0.00001

### 批大小

In [14]:
BATCH_SIZE = 2

### 轮次

**轮次**（Epoch）是指对训练集完整遍历一次。
设为 `1000` 轮，即每个训练样本会被模型学习 `1000` 次。
轮次越多，参数越接近最优值，但过多可能导致过拟合。

In [15]:
EPOCHS = 1000

### 建模

建模流程与上一章相同：分别创建数据集、层、损失函数和优化器，
再将它们作为参数传入模型。

In [16]:
dataset = IcecreamDataset(BATCH_SIZE)

layer = Linear(2, 1)
loss_fn = MSELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)

model = NNModel(layer, loss_fn, optimizer)
print(model.layer)

Linear[weight(1, 2); bias(1,)]


### 迭代训练

现在只需一行代码启动训练，模型会自动完成 1000 轮迭代。

In [17]:
model.train(dataset, EPOCHS)

## 验证

### 测试

训练完成后，一行代码切换到测试模式并评估。

In [18]:
predictions, loss = model.test(dataset)
print(f'prediction: {predictions}')
print(f'test loss:  {loss}')

prediction: Tensor([[163.52327795]])
test loss:  Tensor(2.1807080003283335)


经过 1000 轮训练，模型预测值约为 `163.5`，真实值为 `165`，测试损失约 `2.18`。

至此，我们的神经网络训练框架已经可以完整运作：从原始数据出发，经过多轮批次训练，最终输出接近真实值的预测结果。

## 课后练习

扩展 `NNModel`，增加一个**早停**（Early Stopping）机制：如果测试损失连续 n 轮不再下降，则提前停止训练。